### Fine-Tuning a model to predict labels for LiDAR Point Clouds - 3D Semantic Segmentation - Waymo Open Dataset

#### By Jacob Igo

In [1]:
import gcsfs
from google.cloud import storage
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import tensorflow as tf
import open3d as o3d
import numpy as np
import gc
import os
import random

import semseg_functions


/home/jacob/projects/waymo_research/perception/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jacob/projects/waymo_research/perception/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:17

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:

#get google cloud token
os.environ["CLOUDSDK_CONFIG"] = "/home/jacob/.config/gcloud"

import subprocess
token = subprocess.check_output(
    ["/usr/bin/gcloud", "auth", "print-access-token"]
).decode().strip()

from datetime import datetime, timezone, timedelta
fs = pafs.GcsFileSystem(access_token=token, credential_token_expiration=datetime.now(timezone.utc) + timedelta(hours=1))

folders = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/"))
lidar_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar/"))
seg_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_segmentation/"))
calib_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_calibration/"))


In [3]:
def frame_loader(basefile, timestamp, sample_size=4096):

    calib_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_calibration/{basefile}", filesystem=fs)
    calib_df = calib_pq.read_row_group(0).to_pandas()

    # seg_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", filesystem=fs)
    # lidar_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", filesystem=fs)

    lidar_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", timestamp=timestamp)
    seg_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", timestamp=timestamp)

    g_coords_1, masked_labels_1 = semseg_functions.laser_process(laser_num=1, df=lidar_df, df_rgc=calib_df, df_seg=seg_df, timestamp=timestamp)

    #sampling, as model expects fixed input size
    rng = np.random.default_rng(seed=42)
    random_indices = rng.integers(low=0, high=len(g_coords_1), size=sample_size)
    random_points = g_coords_1[random_indices]
    random_labels = masked_labels_1[random_indices]

    #normalizing, keeps points on a consistent scale for easier learning
    points_min = random_points.min(axis=0)
    points_max = random_points.max(axis=0)

    random_points_norm = (random_points - points_min) / (points_max - points_min)

    return random_points_norm, random_labels





In [ ]:
seg_training = semseg_functions.folder_file_indexer(folder="training/lidar_segmentation/", start_folder_index=0, end_folder_index=2)

def batch_loader(K, training_set, seed=42):
    sample_list = random.sample(training_set, k=K)

    point_clouds = []
    label_sets = []

    for sample in sample_list:

        file_name, timestamp = sample
        base_file_name = os.path.basename(file_name)
        points, labels = frame_loader(basefile=base_file_name, timestamp=timestamp)

        #scanning the labels and points to see if they are valid instances
        unique_labels, label_counts_by_index = np.unique(labels, return_counts=True)
        print("Unique labels per point cloud: ", (len(unique_labels)))
        # print("How many times does each label occur: ", label_counts_by_index)

        if len(unique_labels) < 2:
            raise ValueError("not enough labels")
        
        point_clouds.append(points)
        label_sets.append(labels)

    
    # points_labels = np.vstack((point_clouds, label_sets))
    stacked_points = np.stack(point_clouds)
    stacked_labels = np.stack(label_sets)

    
    return stacked_points, stacked_labels


    

    
def train_val_split(index, train_split=80, seed=42):

    training_file_list = []
    test_file_list = []

    random.seed(seed)
    set_of_files = set()
    
    for file, _ in index:
        base_file = os.path.basename(file)
        set_of_files.add(base_file)

    list_of_files = list(set_of_files)
    shuffled_files = random.sample(list_of_files, len(set_of_files))

    counter=0
    train_split_percentage = train_split / 100
    for unique_file in shuffled_files:
        if counter < len(shuffled_files) * train_split_percentage:
            training_file_list.append(unique_file)
            counter += 1
        else:
            test_file_list.append(unique_file)

    # return training_file_list, test_file_list 
    # adding the frames to the files after splitting for the dataloader
    training_frames = []
    val_frames = []

    for split_file, timestamp in index:
        base_split_file = os.path.basename(split_file)
        if base_split_file in training_file_list:
            training_frames.append((base_split_file, timestamp))
        elif base_split_file in test_file_list:
            val_frames.append((base_split_file, timestamp))
        else:
            raise ValueError("This basefile in the main index was NOT in the training or test file lists")

    return training_frames, val_frames


    


    
training, validating = train_val_split(seg_training)
points, labels = batch_loader(K=5, training_set=training)

print(f"Shape of stacked points: {points.shape}")
print(f"Shape of stacked labels: {labels.shape}")

Unique labels per point cloud:  16
Unique labels per point cloud:  13
Unique labels per point cloud:  12
Unique labels per point cloud:  12
Unique labels per point cloud:  13
Shape of stacked points: (5, 4096, 3)
Shape of stacked labels: (5, 4096)
